In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
import sys
import os

sys.path.append(os.path.abspath('..'))

from src.tree_models import DecisionTreeClassifierCustom

data = {
    'Clima': ['Sol', 'Sol', 'Nublado', 'Lluvia', 'Lluvia', 'Lluvia', 'Nublado', 'Sol', 'Sol', 'Lluvia', 'Sol', 'Nublado', 'Nublado', 'Lluvia'],
    'Temperatura': [85, 80, 83, 70, 68, 65, 64, 72, 69, 75, 75, 72, 81, 71],
    'Viento': ['Suave', 'Fuerte', 'Suave', 'Suave', 'Suave', 'Fuerte', 'Fuerte', 'Suave', 'Suave', 'Suave', 'Fuerte', 'Fuerte', 'Suave', 'Fuerte'],
    'Resultado': ['L', 'L', 'E', 'V', 'V', 'L', 'E', 'L', 'V', 'V', 'E', 'E', 'E', 'L'] 
}
df_toy = pd.DataFrame(data)

X_toy = df_toy.drop('Resultado', axis=1)
y_toy = df_toy['Resultado']

tree_model = DecisionTreeClassifierCustom(min_info_gain=0.01)
tree_model.fit(X_toy, y_toy)

preds = tree_model.predict(X_toy)
print(f"Predicciones: {preds}")
print(f"Precisión en el entrenamiento: {accuracy_score(y_toy, preds)}")

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
# Agora essas importações vão funcionar perfeitamente!
from src.pipeline import get_train_test_data, preprocessing_pipeline
from src.tree_models import DecisionTreeClassifierCustom

print("Cargando y procesando los datos...")
# 1. Cargar y dividir los datos en el tiempo (Train: ->2023 | Test: 2024-2025)
X_train, y_train, X_test, y_test = get_train_test_data(filepath='../data/raw/futbol_uruguayo.csv')

# 2. Aplicar el pipeline de preprocesamiento del Rol 1
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

# 3. Recuperar los nombres de las columnas generadas por el OneHotEncoder
nomi_colonne = preprocessing_pipeline.named_steps['encoder'].get_feature_names_out()

# 4. Convertir a DataFrame (¡Ojo! Ya NO usamos .toarray() porque configuramos sparse_output=False)
X_train_visibile = pd.DataFrame(X_train_processed, columns=nomi_colonne)
X_test_visibile = pd.DataFrame(X_test_processed, columns=nomi_colonne)

print("Entrenando el Árbol de Decisión Personalizado...")
# El entrenamiento debe realizarse hasta 2023 (usando y_train)
arvore_modelo = DecisionTreeClassifierCustom(min_info_gain=0.005)
arvore_modelo.fit(X_train_visibile, y_train)

print("Realizando predicciones para 2024-2025...")
# Evaluando estrictamente en la ventana temporal de prueba
preds_arvore = arvore_modelo.predict(X_test_visibile)

# Calculando las métricas exigidas por la tarea
macro_f1 = f1_score(y_test, preds_arvore, average='macro')
acuracia = accuracy_score(y_test, preds_arvore)

print(f"\nResultados del Árbol de Decisión:")
print(f"Precisión (Accuracy): {acuracia:.4f}")
print(f"Macro-F1 (Métrica Principal): {macro_f1:.4f}")
print("\nMatriz de Confusión:")
# Fijamos el orden de las etiquetas para que 'L', 'E' y 'V' aparezcan siempre en la misma posición
print(confusion_matrix(y_test, preds_arvore, labels=['L', 'E', 'V']))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("Entrenando el Random Forest de Scikit-Learn...")
# Instanciando el modelo. 
# Usamos max_depth para evitar que los árboles crezcan infinitamente (similar al min_info_gain)
rf_modelo = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_modelo.fit(X_train_visibile, y_train)

print("Realizando predicciones para 2024-2025...")
preds_rf = rf_modelo.predict(X_test_visibile)

# Calculando métricas globales
macro_f1_rf = f1_score(y_test, preds_rf, average='macro')
acuracia_rf = accuracy_score(y_test, preds_rf)

print(f"\nResultados del Random Forest:")
print(f"Precisión (Accuracy): {acuracia_rf:.4f}")
print(f"Macro-F1 (Métrica Principal): {macro_f1_rf:.4f}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, preds_rf, labels=['L', 'E', 'V']))

print("\nReporte detallado por clase (Precisión, Recall, F1):")
# Esto genera automáticamente la tabla que pide la letra de la tarea
print(classification_report(y_test, preds_rf, labels=['L', 'E', 'V']))